# 06 - Two-Stage Agent Inference Demo

Runs `product -> reactants/class -> conditions` and converts compact outputs into the app-compatible route JSON schema.

In [1]:
!pip install -q -U transformers peft accelerate bitsandbytes safetensors sentencepiece rdkit

import json
import re
from typing import Optional

import torch
from peft import PeftModel
from rdkit import Chem, RDLogger
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

RDLogger.DisableLog('rdApp.warning')
RDLogger.DisableLog('rdApp.error')

BASE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
REACTANT_LORA_ID = 'oleh13/retro-reactants-qwen25-7b-lora'
CONDITION_LORA_ID = 'oleh13/retro-conditions-qwen25-7b-lora'

REACTANT_SYSTEM = '''You are a chemistry reaction prediction model.
Return valid compact JSON only.
Predict reactants and reaction class for a single-step synthesis of the target product.
Use canonical SMILES strings when possible.
Do not include conditions or explanations.'''

CONDITION_SYSTEM = '''You are a chemistry condition recommendation model.
Return valid compact JSON only.
Given product SMILES, reactants, and reaction class, predict practical reaction conditions.
Do not invent evidence IDs and do not include explanations.'''


In [2]:
def canonical_smiles(smiles: Optional[str]) -> Optional[str]:
    if not smiles:
        return None
    try:
        mol = Chem.MolFromSmiles(str(smiles).strip())
        return Chem.MolToSmiles(mol, canonical=True) if mol is not None else None
    except Exception:
        return None

def extract_json(text: str) -> dict:
    text = text.strip()
    text = re.sub(r'^```(?:json)?', '', text, flags=re.I).strip()
    text = re.sub(r'```$', '', text).strip()
    first, last = text.find('{'), text.rfind('}')
    if first == -1 or last == -1 or last <= first:
        raise ValueError(f'No JSON object found: {text[:300]}')
    return json.loads(text[first:last + 1])

def load_lora(adapter_id: str):
    tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map='auto', torch_dtype=torch.bfloat16, trust_remote_code=True)
    model = PeftModel.from_pretrained(base, adapter_id)
    model.eval()
    return tok, model

def generate(tok, model, messages, max_new_tokens=320):
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok([prompt], return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None, eos_token_id=tok.eos_token_id, pad_token_id=tok.pad_token_id)
    raw = tok.decode(out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()
    return extract_json(raw), raw


In [3]:
react_tok, react_model = load_lora(REACTANT_LORA_ID)
cond_tok, cond_model = load_lora(CONDITION_LORA_ID)

def predict_reactants(product_smiles: str) -> dict:
    target = canonical_smiles(product_smiles)
    if not target:
        raise ValueError(f'Invalid target SMILES: {product_smiles}')
    user = json.dumps({'task': 'predict_reactants_and_class', 'product_smiles': target}, separators=(',', ':'))
    data, raw = generate(react_tok, react_model, [{'role': 'system', 'content': REACTANT_SYSTEM}, {'role': 'user', 'content': user}], max_new_tokens=256)
    reactants = data.get('reactants') if isinstance(data.get('reactants'), list) else []
    valid_reactants = [canonical_smiles(r) for r in reactants]
    valid_reactants = [r for r in valid_reactants if r]
    if not valid_reactants:
        raise ValueError(f'No valid reactants predicted. Raw output: {raw}')
    data['reactants'] = valid_reactants
    data['product_smiles'] = target
    return data

def predict_conditions(product_smiles: str, reactants: list[str], reaction_class: str) -> dict:
    user = json.dumps({'task': 'predict_conditions', 'product_smiles': product_smiles, 'reactants': reactants, 'reaction_class': reaction_class}, separators=(',', ':'))
    data, raw = generate(cond_tok, cond_model, [{'role': 'system', 'content': CONDITION_SYSTEM}, {'role': 'user', 'content': user}], max_new_tokens=320)
    return data

def to_app_route(product_smiles: str, reactant_result: dict, condition_result: dict) -> dict:
    reaction_class = str(reactant_result.get('reaction_class') or 'single-step synthesis')
    return {
        'routes': [{
            'route_name': reaction_class,
            'summary': 'Two-stage fine-tuned model proposal.',
            'steps': [{
                'reaction_name': reaction_class,
                'reactants': reactant_result['reactants'],
                'product_smiles': product_smiles,
                'stoichiometry': 'Reactants as predicted; optimize equivalents experimentally.',
                'reagents': condition_result.get('reagents', 'not specified'),
                'solvent': condition_result.get('solvent', 'not specified'),
                'temperature_celsius': condition_result.get('temperature_celsius', 'not specified'),
                'reaction_time': condition_result.get('reaction_time', 'not specified'),
                'atmosphere': condition_result.get('atmosphere', 'not specified'),
                'workup_purification': condition_result.get('workup_purification', 'not specified'),
                'expected_yield_percent': condition_result.get('expected_yield_percent', 'not specified'),
                'important_conditions': 'Verify route experimentally; model output is a prediction.',
                'rationale': 'Reactants and conditions were generated by separate fine-tuned predictors.',
                'objective_fit': 'Single-step route proposal from target SMILES.',
                'evidence_reaction_ids': [],
            }],
            'objective_fit': 'Single-step route proposal from target SMILES.',
            'evidence_reaction_ids': [],
        }],
        'overall_recommendation': 'Use as a candidate route and validate with literature or experiment.',
    }

def predict_route(product_smiles: str) -> dict:
    target = canonical_smiles(product_smiles)
    reactants = predict_reactants(target)
    conditions = predict_conditions(target, reactants['reactants'], reactants.get('reaction_class', 'unknown'))
    return to_app_route(target, reactants, conditions)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


adapter_config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/323M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

adapter_config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/323M [00:00<?, ?B/s]

In [4]:
test_targets = [
    'N#CCc1ccccc1',
    'O=[N+]([O-])c1ccccc1',
    'CC(C)(O)c1ccccc1',
    'c1ccc(-c2ccccc2)cc1',
    'O=C(/C=C/c1ccccc1)c1ccccc1',
    'C=Cc1ccccc1',
    'C1=CCCCC1',
    'CC(O)c1ccccc1',
    'O=C1OC(=O)C2CC=CCC12',
    'CCNC(C)=O',
]

for target in test_targets:
    print('\nTARGET:', target)
    try:
        result = predict_route(target)
        print(json.dumps(result, indent=2, ensure_ascii=False)[:4000])
    except Exception as exc:
        print('FAILED GRACEFULLY:', exc)



TARGET: N#CCc1ccccc1


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


{
  "routes": [
    {
      "route_name": "9",
      "summary": "Two-stage fine-tuned model proposal.",
      "steps": [
        {
          "reaction_name": "9",
          "reactants": [
            "NC(=O)Cc1ccccc1"
          ],
          "product_smiles": "N#CCc1ccccc1",
          "stoichiometry": "Reactants as predicted; optimize equivalents experimentally.",
          "reagents": "not specified",
          "solvent": "acetic anhydride",
          "temperature_celsius": "not specified",
          "reaction_time": "not specified",
          "atmosphere": "not specified",
          "workup_purification": "TEMPERATURE: The mixture was heated; TEMPERATURE: at reflux for 2 hours; CUSTOM: The solvent was removed in vacuo; ADDITION: the residue treated with water (50 ml); EXTRACTION: extracted with ether (3×50 ml); WASH: The combined organic extracts were washed with brine; DRY_WITH_MATERIAL: dried over sodium sulfate; CONCENTRATION: concentrated in vacuo",
          "expected_yield_perce